# Stack Doctor — GRPO Training with Unsloth + TRL

Train Qwen2.5-1.5B to diagnose GPU inference stack failures using GRPO.
Built on [OpenEnv](https://github.com/openenv-ai/openenv).

In [ ]:
!pip install -q unsloth trl openenv-core weave
!git clone https://github.com/bledden/stack-doctor.git /content/stack-doctor
import sys; sys.path.insert(0, "/content/stack-doctor")

In [ ]:
import json, random, re
import weave
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

from server.stack_doctor_environment import StackDoctorEnvironment
from server.scenarios import HANDCRAFTED_TRAIN, SCRAPED_TRAIN_SCENARIOS
from models import StackDoctorAction

weave.init("stack-doctor-grpo")

In [ ]:
SYSTEM_PROMPT = """You are Stack Doctor, an expert AI agent that diagnoses inference-stack incidents.
You receive an incident ticket with hardware/model/backend context, log excerpts, and specialist opinions.
Some specialists may be wrong. Output a JSON array of actions:
  {"type":"inspect","target":"logs|config|snippet|metrics"}
  {"type":"ask_specialist","specialist":"runtime|dispatch|kernel|loader"}
  {"type":"apply_fix","fix":"<fix_name>"}
  {"type":"submit","root_cause":"<cause>","fix":"<fix>","justification":"<why>"}"""

def format_scenario_prompt(sc):
    ops = {n: f"{o.opinion} (confidence: {o.confidence})" for n, o in sc.specialist_opinions.items()}
    return f"""INCIDENT: {sc.incident_ticket}
Hardware: {sc.hardware} | Model: {sc.model_name} | Backend: {sc.backend}
LOG:
{sc.initial_log}
SPECIALISTS:
{json.dumps(ops, indent=2)}
Investigate and submit your diagnosis as a JSON action array."""

def extract_actions(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    start, end = text.find("["), text.rfind("]")
    if start != -1 and end > start:
        try:
            actions = json.loads(text[start:end+1])
            if isinstance(actions, list): return [a for a in actions if isinstance(a, dict)] or None
        except: pass
    try:
        actions = json.loads(text)
        if isinstance(actions, list): return [a for a in actions if isinstance(a, dict)] or None
        if isinstance(actions, dict): return [actions]
    except: pass
    return None

In [ ]:
def valid_json_reward(completions, **kw):
    scores = []
    for c in completions:
        r = c[0]["content"] if isinstance(c, list) else c
        a = extract_actions(r)
        if a is None: scores.append(-3.0)
        elif not any(x.get("type")=="submit" for x in a): scores.append(-1.0)
        else: scores.append(1.0)
    return scores

def environment_reward(completions, **kw):
    scores, sids = [], kw.get("scenario_id", [None]*len(completions))
    for i, c in enumerate(completions):
        r = c[0]["content"] if isinstance(c, list) else c
        actions = extract_actions(r)
        if actions is None: scores.append(-5.0); continue
        env = StackDoctorEnvironment()
        env.reset(scenario_id=sids[i] if i < len(sids) else None)
        cum = 0.0
        for ad in actions:
            if not isinstance(ad, dict): cum -= 2.0; continue
            try:
                obs = env.step(StackDoctorAction(message=json.dumps(ad)))
                cum += obs.reward
                if obs.done: break
            except: cum -= 2.0; break
        scores.append(cum)
    return scores

def investigation_quality_reward(completions, **kw):
    scores = []
    for c in completions:
        r = c[0]["content"] if isinstance(c, list) else c
        actions = extract_actions(r)
        if actions is None: scores.append(0.0); continue
        inv = sum(1 for a in actions if isinstance(a,dict) and a.get("type") in ("inspect","ask_specialist"))
        has_sub = any(isinstance(a,dict) and a.get("type")=="submit" for a in actions)
        if not has_sub: scores.append(0.0)
        elif inv == 0: scores.append(-1.0)
        elif inv <= 2: scores.append(0.5)
        else: scores.append(1.0)
    return scores

In [ ]:
def build_dataset(scenarios, n_repeats=10):
    data = []
    for sc in scenarios:
        for _ in range(n_repeats):
            data.append({"prompt": [{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":format_scenario_prompt(sc)}], "scenario_id": sc.id})
    random.shuffle(data)
    return data

train_data = build_dataset(HANDCRAFTED_TRAIN, 30) + build_dataset(SCRAPED_TRAIN_SCENARIOS, 5)
random.shuffle(train_data)
dataset = Dataset.from_list(train_data)
print(f"Dataset: {len(dataset)} episodes")

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct", load_in_4bit=True, max_seq_length=8192)
model = FastLanguageModel.get_peft_model(model, r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=32, use_gradient_checkpointing="unsloth")

In [ ]:
training_args = GRPOConfig(
    temperature=1.0, learning_rate=5e-5, weight_decay=0.001,
    warmup_ratio=0.1, lr_scheduler_type="linear", optim="adamw_8bit",
    logging_steps=1, per_device_train_batch_size=1,
    gradient_accumulation_steps=1, num_generations=2,
    max_prompt_length=4096, max_completion_length=1024,
    max_steps=50, save_steps=10, output_dir="outputs")

trainer = GRPOTrainer(
    model=model, processing_class=tokenizer,
    reward_funcs=[valid_json_reward, environment_reward, investigation_quality_reward],
    args=training_args, train_dataset=dataset)

trainer.train()
model.save_pretrained("stack_doctor_lora")
print("Done!")